# Experiment 7: Label Smoothing CrossEntropyLoss

**Single variable changed**: loss function — `CrossEntropyLoss(label_smoothing=0.1)`
**Held constant**: architecture (DiagnosticCNN), data (no augmentation), optimizer (Adam lr=0.001), epochs (15)

## Rationale

Our error analysis showed misclassification confidence is ~0.76 — mid-range, not random. The model is "torn" between classes, but the winner-take-all dynamic of hard one-hot targets forces it to pick one. Label smoothing relaxes targets from `[0, 1]` to `[0.1/9, 0.9]`, reducing the penalty for non-maximum classes. This may break the zero-sum sink by letting the model distribute probability mass more evenly among similar upper-body classes without sharp winner-take-all decisions.

In [1]:
import sys
sys.path.append('..')

import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
from src.data_utils import load_fashionmnist, get_dataloaders
from src.train_utils import train_one_epoch
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores
)

OUT_DIR = '../outputs/error_analysis/label_smoothing'
os.makedirs(OUT_DIR, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
if torch.backends.mps.is_available():    device = 'mps'
elif torch.cuda.is_available():          device = 'cuda'
else:                                    device = 'cpu'
print(f"Device: {device}")

PyTorch: 2.13.0+cu130
Device: cuda


## Dataset — identical to E1 (no augmentation)

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_dataset = __import__('torchvision').datasets.FashionMNIST(
    root='../data', train=True, download=True, transform=transform
)
test_dataset = __import__('torchvision').datasets.FashionMNIST(
    root='../data', train=False, download=True, transform=transform
)

class_names = train_dataset.classes
train_loader, test_loader = get_dataloaders(train_dataset, test_dataset, batch_size=64)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

Train batches: 938, Test batches: 157


## Architecture — identical to E1 DiagnosticCNN

In [3]:
class DiagnosticCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, num_classes)
        self.relu = nn.ReLU(inplace=True)

    def get_features(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.pool3(x)
        x = self.global_pool(x)
        return x.view(x.size(0), -1)

    def forward(self, x):
        x = self.get_features(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = DiagnosticCNN().to(device)
print(f"DiagnosticCNN params: {sum(p.numel() for p in model.parameters()):,}")

DiagnosticCNN params: 140,778


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## Loss — CrossEntropy with label smoothing (the only change)

In [4]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
print(f"Criterion: {criterion}")

optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 15

Criterion: CrossEntropyLoss()


## Training

In [5]:
train_losses = []
model.train()
for epoch in range(num_epochs):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss:.4f}')

with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for loss in train_losses:
        f.write(f'{loss}\n')
print(f"Losses saved.")

Epoch [1/15], Loss: 0.8916
Epoch [2/15], Loss: 0.7672
Epoch [3/15], Loss: 0.7326
Epoch [4/15], Loss: 0.7113
Epoch [5/15], Loss: 0.6967
Epoch [6/15], Loss: 0.6866
Epoch [7/15], Loss: 0.6738
Epoch [8/15], Loss: 0.6651
Epoch [9/15], Loss: 0.6562
Epoch [10/15], Loss: 0.6495
Epoch [11/15], Loss: 0.6419
Epoch [12/15], Loss: 0.6360
Epoch [13/15], Loss: 0.6282
Epoch [14/15], Loss: 0.6249
Epoch [15/15], Loss: 0.6185
Losses saved.


## Evaluation — identical pipeline

In [6]:
accuracy, cm, per_class = evaluate_detailed(
    model, test_loader, device, class_names, model_name='SmoothCNN'
)
probas, labels = get_all_probas_and_labels(model, test_loader, device, 10)
roc_scores = compute_roc_auc_scores(probas, labels, model_name='SmoothCNN')
pr_scores = compute_pr_auc_scores(probas, labels, model_name='SmoothCNN')

with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')
    f.write(f'Macro ROC-AUC: {roc_scores["macro"]:.6f}\n')
    f.write(f'Macro PR-AUC:  {pr_scores["macro"]:.6f}\n\n')
    f.write(f'{"Class":<15} {"ROC-AUC":>10} {"PR-AUC":>10} {"TPR":>10} {"Precision":>10}\n')
    f.write('-' * 55 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']
        prec = per_class[name]['Precision']
        f.write(f'{name:<15} {roc_scores[f"class_{i}"]:>10.4f} {pr_scores[f"class_{i}"]:>10.4f} {tpr:>10.4f} {prec:>10.4f}\n')

cm_np = cm.cpu().numpy()
with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names:
        f.write(f'{name:>15}')
    f.write('\n')
    for i in range(len(class_names)):
        f.write(f'{class_names[i]:>15}')
        for j in range(len(class_names)):
            f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:
    f.write('Misclassification Analysis\n')
    f.write('=' * 70 + '\n\n')
    for c in range(len(class_names)):
        true_name = class_names[c]
        total_errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {true_name}  (errors: {total_errors})\n')
        f.write('-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0:
                continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')

print(f"\nAll results saved to {OUT_DIR}/")

  Test Accuracy: 92.04%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8390     0.0094     0.9080
  Trouser             0.9730     0.0003     0.9969
  Pullover            0.7900     0.0030     0.9670
  Dress               0.9380     0.0107     0.9072
  Coat                0.8740     0.0116     0.8937
  Sandal              0.9760     0.0007     0.9939
  Shirt               0.8790     0.0434     0.6921
  Sneaker             0.9900     0.0064     0.9447
  Bag                 0.9880     0.0012     0.9890
  Ankle boot          0.9570     0.0017     0.9846

All results saved to ../outputs/error_analysis/label_smoothing/


## Delta vs E1 Baseline

In [7]:
def load_e1_metrics(path):
    data = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5 and parts[0] != 'Class' and '-' not in line[:5]:
                cls, roc, pr, tpr, prec = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                data[cls] = {'tpr': tpr, 'precision': prec, 'pr_auc': pr}
    return data

e1 = load_e1_metrics('../outputs/error_analysis/metrics_summary.txt')

print(f'{"Class":<15} {"E1 TPR":>8} {"E7 TPR":>8} {"Δ TPR":>8} {"E1 Prec":>8} {"E7 Prec":>8} {"Δ Prec":>8}')
print('-' * 63)
for name in class_names:
    if name in e1:
        print(f'{name:<15} {e1[name]["tpr"]:>8.3f} {per_class[name]["TPR"]:>8.3f} {per_class[name]["TPR"] - e1[name]["tpr"]:>+8.3f} {e1[name]["precision"]:>8.3f} {per_class[name]["Precision"]:>8.3f} {per_class[name]["Precision"] - e1[name]["precision"]:>+8.3f}')

print(f'\nAccuracy:  E1=92.50%  E7={accuracy:.2f}%  Δ={accuracy - 92.50:+.2f}%')
print(f'Macro PR:   E1=0.9712  E7={pr_scores["macro"]:.4f}  Δ={pr_scores["macro"] - 0.9712:+.4f}')

e7_shirt_err = cm_np[6].sum() - cm_np[6, 6]
e7_tshirt_err = cm_np[0].sum() - cm_np[0, 0]
e7_upper = e7_shirt_err + e7_tshirt_err + (cm_np[2].sum()-cm_np[2,2]) + (cm_np[4].sum()-cm_np[4,4]) + (cm_np[3].sum()-cm_np[3,3])
print(f'\nUpper-body total errors: E1=649, E7={e7_upper}')
print(f'Shirt errors:   E1=153, E7={e7_shirt_err}')
print(f'T-shirt errors: E1=163, E7={e7_tshirt_err}')

Class             E1 TPR   E7 TPR    Δ TPR  E1 Prec  E7 Prec   Δ Prec
---------------------------------------------------------------
Trouser            0.988    0.973   -0.015    0.990    0.997   +0.007
Pullover           0.890    0.790   -0.100    0.896    0.967   +0.071
Dress              0.907    0.938   +0.031    0.940    0.907   -0.033
Coat               0.870    0.874   +0.004    0.931    0.894   -0.038
Sandal             0.978    0.976   -0.002    0.990    0.994   +0.004
Shirt              0.847    0.879   +0.032    0.723    0.692   -0.031
Sneaker            0.991    0.990   -0.001    0.946    0.945   -0.001
Bag                0.986    0.988   +0.002    0.982    0.989   +0.007

Accuracy:  E1=92.50%  E7=92.04%  Δ=-0.46%
Macro PR:   E1=0.9712  E7=0.9709  Δ=-0.0003

Upper-body total errors: E1=649, E7=680
Shirt errors:   E1=153, E7=121
T-shirt errors: E1=163, E7=161
